In [93]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats import kruskal


In [2]:
data = pd.read_csv("..\datasets\diabetic_data_initial.csv")

In [3]:
def data_overview(x):
 pd.set_option("display.max_columns",50)
 pd.set_option("display.max_rows", None)
 display(x.head(50))
 print(f"Number of rows:{x.shape[0]}\nNumber of columns:{x.shape[1]}")
 print(pd.DataFrame({
     "Data type:": x.dtypes,
     "Missing values:": x.isna().sum(), 
     "Percentage of na:": x.isna().mean()*100,
     "Value count:":  x.nunique()
 }))


In [4]:
data_overview(data)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
5,35754,82637451,Caucasian,Male,[50-60),?,2,1,2,3,?,?,31,6,16,0,0,0,414,411,250,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
6,55842,84259809,Caucasian,Male,[60-70),?,3,1,2,4,?,?,70,1,21,0,0,0,414,411,V45,7,NaN,NaN,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
7,63768,114882984,Caucasian,Male,[70-80),?,1,1,7,5,?,?,73,0,12,0,0,0,428,492,250,8,NaN,NaN,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
8,12522,48330783,Caucasian,Female,[80-90),?,2,1,4,13,?,?,68,2,28,0,0,0,398,427,38,8,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,15738,63555939,Caucasian,Female,[90-100),?,3,3,4,12,?,InternalMedicine,33,3,18,0,0,0,434,198,486,8,NaN,NaN,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Number of rows:101766
Number of columns:50
                         Data type:  Missing values:  Percentage of na:  \
encounter_id                  int64                0           0.000000   
patient_nbr                   int64                0           0.000000   
race                         object                0           0.000000   
gender                       object                0           0.000000   
age                          object                0           0.000000   
weight                       object                0           0.000000   
admission_type_id             int64                0           0.000000   
discharge_disposition_id      int64                0           0.000000   
admission_source_id           int64                0           0.000000   
time_in_hospital              int64                0           0.000000   
payer_code                   object                0           0.000000   
medical_specialty            object                0     

U skupu podataka uocava se string "?" koji predstavlja nedostajucu vrednost, ali je python tako ne prepoznaje pa ce "?" biti zamenjen sa NaN

In [5]:
def replacing_with_na(x):
   missing_value = "?"
   for column in x.columns:
        x[column]= x[column].replace(missing_value, np.nan)
   return x

In [6]:
data = replacing_with_na(data)

Kolone "weight" i "max_glu_serum" imaju visok procenat nedostajucih vrednosti(>94%), ali ce pre njihovog izbacivanja biti ispitano da li postoji neka zakonitost u podacima zbog koje je izostavljen veci deo ovih podataka.  

In [7]:
data["weight"].describe()

count         3197
unique           9
top       [75-100)
freq          1336
Name: weight, dtype: object

In [8]:
data["max_glu_serum"].describe()

count     5346
unique       3
top       Norm
freq      2597
Name: max_glu_serum, dtype: object

Analizom distribucije dostupnih vrednosti za atribute weight i max_glu_serum nije uoceno da su podaci belezeni iskljucivo za ekstremne slucajeve. Najzastupljenije kategorije medju postojecim vrednostima odgovaraju uobicajenim vrednostima, sto ukazuje da nedostajanje ovih podataka verovatno nije posledica selektivnog belezenja samo kod pacijenata sa ekstremnim vrednostima.

In [9]:
def compare_column_with_all_columns(name_of_column):
 pd.set_option("display.max_columns", None)
 for column in data.columns:
    if column != name_of_column:
        if data[column].dtype == "object" and data[name_of_column].dtype=="object":
          display(pd.crosstab(data[name_of_column], data[column], normalize="index") *100)
        else:
            print(f"\n--- {column} ---")
            display(data.groupby(name_of_column)[column].describe())

In [10]:
data["has_weight"]= data["weight"].notna()
data["has_weight"].dtype
data["has_weight"]= data["has_weight"].astype("object")


In [11]:
compare_column_with_all_columns("has_weight")
data["has_weight"].value_counts()


--- encounter_id ---


,count,mean,std,min,25%,50%,75%,max
has_weight,,,,,,,,
False,98569.0,1.655019e+08,1.026795e+08,12522.0,85035102.0,152907534.0,230816256.0,443867222.0
True,3197.0,1.559442e+08,1.010020e+08,8042082.0,83230266.0,127781592.0,212779206.0,443695760.0



--- patient_nbr ---


,count,mean,std,min,25%,50%,75%,max
has_weight,,,,,,,,
False,98569.0,5.378262e+07,3.879460e+07,135.0,23368149.0,44297208.0,87170085.0,189502619.0
True,3197.0,7.121929e+07,3.112114e+07,1629.0,54222390.0,76705218.0,95722092.0,186768005.0


race,AfricanAmerican,Asian,Caucasian,Hispanic,Other
has_weight,,,,,
False,19.803181,0.655371,75.898542,2.112326,1.530581
True,3.694018,0.294214,95.031056,0.000000,0.980713


...

metformin-pioglitazone,No,Steady
has_weight,,
False,99.998985,0.001015
True,100.000000,0.000000


change,Ch,No
has_weight,,
False,46.612018,53.387982
True,33.343760,66.656240


diabetesMed,No,Yes
has_weight,,
False,22.732299,77.267701
True,31.154207,68.845793


readmitted,<30,>30,NO
has_weight,,,
False,11.159695,34.529112,54.311193
True,11.166719,47.231780,41.601501


has_weight
False    98569
True      3197
Name: count, dtype: int64

Analizom atributa „weight“ uoceno je da nedostajanje vrednosti varira u odnosu na pojedine kategorije drugih atributa, pri cemu je izrazenije kod pacijenata rase African American, bez insulinske terapije, bez promene terapije, kao i kod pacijenata koji nisu ponovo hospitalizovani. Medjutim, uoceni obrasci nisu dovoljno konzistentni za donosenje pouzdanih zakljucaka o povezanosti nedostajanja sa ovim karakteristikama. Posebno je uocena veca ucestalost nedostajanja kod pacijenata koji nisu ponovo hospitalizovani, sto nije intuitivan obrazac ukoliko se ocekuje veca dostupnost podataka kod pacijenata sa ponovnim kontaktom sa zdravstvenim sistemom. Zbog toga se visok procenat nedostajucih vrednosti pre pripisuje nacinu prikupljanja podataka u periodu 1999–2008. godine, kada zdravstvene ustanove nisu bile obavezne da telesnu tezinu evidentiraju u strukturisanom elektronskom obliku. Na osnovu navedenog, atribut „weight“ je uklonjen iz dalje analize.

In [12]:
data = data.drop(["weight","has_weight"], axis= 1)

In [13]:
data["has_max_glu_serum"]= data["max_glu_serum"].notna()
data["has_max_glu_serum"]= data["has_max_glu_serum"].astype("object")

In [14]:
compare_column_with_all_columns("has_max_glu_serum")


--- encounter_id ---


,count,mean,std,min,25%,50%,75%,max
has_max_glu_serum,,,,,,,,
False,96420.0,1.690902e+08,1.028127e+08,12522.0,89035549.5,155899620.0,234767727.0,443867222.0
True,5346.0,9.506861e+07,6.859692e+07,2265486.0,55859011.5,82505202.0,104741811.0,439020524.0



--- patient_nbr ---


,count,mean,std,min,25%,50%,75%,max
has_max_glu_serum,,,,,,,,
False,96420.0,5.522301e+07,3.905976e+07,135.0,23273201.25,49196052.0,88476435.0,189502619.0
True,5346.0,3.823136e+07,2.672763e+07,209394.0,23637393.00,24455736.0,29903877.0,174774587.0


race,AfricanAmerican,Asian,Caucasian,Hispanic,Other
has_max_glu_serum,,,,,
False,19.905948,0.652845,75.951934,1.964906,1.524368
True,8.657845,0.491493,86.011342,3.516068,1.323251


...

metformin-rosiglitazone,No,Steady
has_max_glu_serum,,
False,99.997926,0.002074
True,100.000000,0.000000


metformin-pioglitazone,No,Steady
has_max_glu_serum,,
False,99.998963,0.001037
True,100.000000,0.000000


change,Ch,No
has_max_glu_serum,,
False,46.311968,53.688032
True,44.089039,55.910961


diabetesMed,No,Yes
has_max_glu_serum,,
False,22.806472,77.193528
True,26.430976,73.569024


readmitted,<30,>30,NO
has_max_glu_serum,,,
False,11.093134,34.856876,54.049990
True,12.364385,36.213992,51.421624


Analizom atributa „has_max_glu_serum“ ispitano je da li dostupnost rezultata serumске glukoze pokazuje povezanost sa ostalim relevantnim atributima. Uoceno je da se nedostajanje ovog podatka ne razlikuje znacajno i konzistentno izmedju analiziranih kategorija, zbog cega sama informacija o tome da li je test evidentiran ne pruza dovoljno osnova za dalje zakljucivanje. Stoga je atribut „has_max_glu_serum“ uklonjen iz dalje analize. Medjutim, atribut „max_glu_serum“ je zadrzan, jer njegove vrednosti predstavljaju rezultat merenja serumske glukoze (Norm, >200, >300) i mogu sadrzati znacajnu informaciju o zdravstvenom stanju pacijenta i njegovoj povezanosti sa ishodom hospitalizacije.

In [15]:
data = data.drop(["has_max_glu_serum"], axis= 1)

In [16]:
data_overview(data)

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
5,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,NaN,NaN,31,6,16,0,0,0,414,411,250,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
6,55842,84259809,Caucasian,Male,[60-70),3,1,2,4,NaN,NaN,70,1,21,0,0,0,414,411,V45,7,NaN,NaN,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
7,63768,114882984,Caucasian,Male,[70-80),1,1,7,5,NaN,NaN,73,0,12,0,0,0,428,492,250,8,NaN,NaN,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
8,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,NaN,NaN,68,2,28,0,0,0,398,427,38,8,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,15738,63555939,Caucasian,Female,[90-100),3,3,4,12,NaN,InternalMedicine,33,3,18,0,0,0,434,198,486,8,NaN,NaN,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Number of rows:101766
Number of columns:49
                         Data type:  Missing values:  Percentage of na:  \
encounter_id                  int64                0           0.000000   
patient_nbr                   int64                0           0.000000   
race                         object             2273           2.233555   
gender                       object                0           0.000000   
age                          object                0           0.000000   
admission_type_id             int64                0           0.000000   
discharge_disposition_id      int64                0           0.000000   
admission_source_id           int64                0           0.000000   
time_in_hospital              int64                0           0.000000   
payer_code                   object            40256          39.557416   
medical_specialty            object            49949          49.082208   
num_lab_procedures            int64                0     

In [17]:
data_id_mapping = pd.read_csv("../datasets/id_mapping.csv")

In [18]:
data_overview(data_id_mapping)

,ID mappings for the diabetes dataset,? - always means a missing value
0,admission_type_id,description
1,1,Emergency
2,2,Urgent
3,3,Elective
4,4,Newborn
5,5,Not Available
6,6,NaN
7,7,Trauma Center
8,8,Not Mapped
9,NaN,NaN


Number of rows:68
Number of columns:2
                                     Data type:  Missing values:  \
ID mappings for the diabetes dataset     object                2   
 ? - always means a missing value        object                5   

                                      Percentage of na:  Value count:  
ID mappings for the diabetes dataset           2.941176            33  
 ? - always means a missing value              7.352941            56  


Pored standardnih nedostajucih vrednosti potrebno je analizirati i tabelu id_mapping.csv i proveriti da li postoje sifre koje se takodje mogu tretirati kao NaN.

In [19]:
data_id_mapping.columns

Index(['ID mappings for the diabetes dataset', ' ? - always means a missing value'], dtype='object')

In [20]:
admission_mapping = data_id_mapping[0:9]

In [21]:
admission_mapping.head(12)

,ID mappings for the diabetes dataset,? - always means a missing value
0,admission_type_id,description
1,1,Emergency
2,2,Urgent
3,3,Elective
4,4,Newborn
5,5,Not Available
6,6,NaN
7,7,Trauma Center
8,8,Not Mapped


In [22]:
admission_mapping.columns= admission_mapping.iloc[0]

In [23]:
admission_mapping= admission_mapping[1:].reset_index(drop=True)

In [24]:
discharge_disposition_mapping = data_id_mapping[10:41]

In [25]:
discharge_disposition_mapping.head(40)

,ID mappings for the diabetes dataset,? - always means a missing value
10,discharge_disposition_id,description
11,1,Discharged to home
12,2,Discharged/transferred to another short term h...
13,3,Discharged/transferred to SNF
14,4,Discharged/transferred to ICF
15,5,Discharged/transferred to another type of inpa...
16,6,Discharged/transferred to home with home healt...
17,7,Left AMA
18,8,Discharged/transferred to home under care of H...
19,9,Admitted as an inpatient to this hospital


In [26]:
discharge_disposition_mapping.columns = discharge_disposition_mapping.iloc[0]

In [27]:
discharge_disposition_mapping = discharge_disposition_mapping[1:].reset_index(drop = True)

In [28]:
discharge_disposition_mapping.columns

Index(['discharge_disposition_id', 'description'], dtype='object', name=10)

In [29]:
discharge_disposition_mapping.columns.name = None

In [30]:
admission_source_mapping = data_id_mapping[42:]

In [31]:
admission_source_mapping.head(30)

,ID mappings for the diabetes dataset,? - always means a missing value
42,admission_source_id,description
43,1,Physician Referral
44,2,Clinic Referral
45,3,HMO Referral
46,4,Transfer from a hospital
47,5,Transfer from a Skilled Nursing Facility (SNF)
48,6,Transfer from another health care facility
49,7,Emergency Room
50,8,Court/Law Enforcement
51,9,Not Available


In [32]:
admission_source_mapping.columns = admission_source_mapping.iloc[0]

In [33]:
admission_source_mapping = admission_source_mapping[1:].reset_index(drop=True)

In [34]:
admission_source_mapping.columns.name = None

In [35]:
def data_frame_to_dict(x):
 dictionary = dict(
 zip(
    x[x.columns[0]].astype(int),
    x[x.columns[1]]
 )  
 )
 return dictionary

In [36]:
admission_dict = data_frame_to_dict(admission_mapping)

In [37]:
discharge_dict  =data_frame_to_dict(discharge_disposition_mapping)

In [38]:
admission_source_dict = data_frame_to_dict(admission_source_mapping)

In [39]:
data_raw = data.copy()

In [40]:
data["admission_type"]= data["admission_type_id"].map(admission_dict)
data["discharge_disposition"] = data["discharge_disposition_id"].map(discharge_dict)
data["admmission_source"]= data["admission_source_id"].map(admission_source_dict)

In [41]:
data_overview(data)

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,...,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type,discharge_disposition,admmission_source
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,NaN,Not Mapped,Physician Referral
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,Emergency,Discharged to home,Emergency Room
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,...,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,...,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
5,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,NaN,NaN,31,6,16,0,0,0,414,411,250,9,NaN,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30,Urgent,Discharged to home,Clinic Referral
6,55842,84259809,Caucasian,Male,[60-70),3,1,2,4,NaN,NaN,70,1,21,0,0,0,414,411,V45,7,NaN,NaN,Steady,No,...,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Elective,Discharged to home,Clinic Referral
7,63768,114882984,Caucasian,Male,[70-80),1,1,7,5,NaN,NaN,73,0,12,0,0,0,428,492,250,8,NaN,NaN,No,No,...,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30,Emergency,Discharged to home,Emergency Room
8,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,NaN,NaN,68,2,28,0,0,0,398,427,38,8,NaN,NaN,No,No,...,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Urgent,Discharged to home,Transfer from a hospital
9,15738,63555939,Caucasian,Female,[90-100),3,3,4,12,NaN,InternalMedicine,33,3,18,0,0,0,434,198,486,8,NaN,NaN,No,No,...,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Elective,Discharged/transferred to SNF,Transfer from a hospital


Number of rows:101766
Number of columns:52
                         Data type:  Missing values:  Percentage of na:  \
encounter_id                  int64                0           0.000000   
patient_nbr                   int64                0           0.000000   
race                         object             2273           2.233555   
gender                       object                0           0.000000   
age                          object                0           0.000000   
admission_type_id             int64                0           0.000000   
discharge_disposition_id      int64                0           0.000000   
admission_source_id           int64                0           0.000000   
time_in_hospital              int64                0           0.000000   
payer_code                   object            40256          39.557416   
medical_specialty            object            49949          49.082208   
num_lab_procedures            int64                0     

Svaki ID koji je u mapping tabeli bio označen kao NULL, zamenjen je vrednošću NaN u kolonama admission_type_id, discharge_disposition_id i admission_source_id.

Kako bi se atribut max_glu_serum adekvatnije procenio, potrebno je uzeti u obzir da nedostajuca vrednost ne mora predstavljati samo odsustvo podatka, vec moze nositi dodatnu informaciju. Na primer, cinjenica da laboratorijski test nije narucen za odredjenog pacijenta moze biti povezana sa karakteristikama pacijenta ili tokom njegovog lecenja. Zbog toga ce atribut max_glu_serum za sada biti detaljnije analiziran pre donosenja konacne odluke o njegovom uklanjanju ili zadrzavanju.

In [42]:
data["max_glu_serum"]= data["max_glu_serum"].fillna("not_measured")
data["max_glu_serum"].unique()
data["max_glu_serum"].value_counts(dropna=True)


max_glu_serum
not_measured    96420
Norm             2597
>200             1485
>300             1264
Name: count, dtype: int64

In [43]:
compare_column_with_all_columns("max_glu_serum")


--- encounter_id ---


,count,mean,std,min,25%,50%,75%,max
max_glu_serum,,,,,,,,
>200,1485.0,8.652990e+07,5.927419e+07,2521974.0,54621210.0,79662234.0,100232430.0,439020524.0
>300,1264.0,1.141084e+08,8.578641e+07,2265486.0,56707584.0,87266640.0,162983068.5,434911802.0
Norm,2597.0,9.068418e+07,6.214560e+07,2309268.0,56106606.0,82728468.0,103001604.0,436664606.0
not_measured,96420.0,1.690902e+08,1.028127e+08,12522.0,89035549.5,155899620.0,234767727.0,443867222.0



--- patient_nbr ---


,count,mean,std,min,25%,50%,75%,max
max_glu_serum,,,,,,,,
>200,1485.0,3.580077e+07,2.487625e+07,530577.0,23608431.00,24241932.0,28428084.0,115186365.0
>300,1264.0,4.717338e+07,3.142402e+07,302238.0,23797289.25,25892923.5,84428613.0,174774587.0
Norm,2597.0,3.526898e+07,2.419855e+07,209394.0,23605038.00,24239133.0,28371474.0,124870523.0
not_measured,96420.0,5.522301e+07,3.905976e+07,135.0,23273201.25,49196052.0,88476435.0,189502619.0


race,AfricanAmerican,Asian,Caucasian,Hispanic,Other
max_glu_serum,,,,,
>200,8.117326,0.477490,86.084584,3.751705,1.568895
>300,9.497207,0.718276,84.596967,3.910615,1.276935
Norm,8.556982,0.388954,86.658888,3.189420,1.205757
not_measured,19.905948,0.652845,75.951934,1.964906,1.524368


...

diabetesMed,No,Yes
max_glu_serum,,
>200,18.653199,81.346801
>300,17.721519,82.278481
Norm,35.117443,64.882557
not_measured,22.806472,77.193528


readmitted,<30,>30,NO
max_glu_serum,,,
>200,12.457912,35.690236,51.851852
>300,14.319620,41.218354,44.462025
Norm,11.359261,34.077782,54.562957
not_measured,11.093134,34.856876,54.049990


admission_type,Elective,Emergency,Newborn,Not Available,Not Mapped,Trauma Center,Urgent
max_glu_serum,,,,,,,
>200,0.097561,14.243902,0.000000,80.878049,0.000000,0.000000,4.780488
>300,0.641026,35.683761,0.000000,59.508547,0.000000,0.000000,4.166667
Norm,0.844773,19.746568,0.000000,72.703273,0.000000,0.000000,6.705385
not_measured,20.347657,57.369899,0.010797,2.183114,0.345498,0.022673,19.720363


discharge_disposition,Admitted as an inpatient to this hospital,Discharged to home,Discharged/transferred to ICF,Discharged/transferred to SNF,Discharged/transferred to a federal health care facility.,Discharged/transferred to a long term care hospital.,Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.,Discharged/transferred to another rehab fac including rehab units of a hospital .,Discharged/transferred to another short term hospital,Discharged/transferred to another type of inpatient care institution,Discharged/transferred to home under care of Home IV provider,Discharged/transferred to home with home health service,Discharged/transferred within this institution to Medicare approved swing bed,Discharged/transferred/referred another institution for outpatient services,Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital,Discharged/transferred/referred to this institution for outpatient services,Expired,"Expired at home. Medicaid only, hospice.","Expired in a medical facility. Medicaid only, hospice.",Hospice / home,Hospice / medical facility,Left AMA,Neonate discharged to another hospital for neonatal aftercare,Not Mapped,Still patient or expected to return for outpatient services
max_glu_serum,,,,,,,,,,,,,,,,,,,,,,,,,
>200,0.000000,35.892256,0.067340,27.609428,0.000000,0.067340,0.000000,0.673401,4.915825,1.548822,0.134680,15.353535,0.00000,0.134680,0.000000,0.202020,3.030303,0.000000,0.000000,0.269360,1.144781,0.606061,0.202020,8.148148,0.000000
>300,0.000000,42.642405,0.237342,21.360759,0.000000,0.000000,0.000000,0.474684,3.639241,1.661392,0.158228,17.088608,0.00000,0.079114,0.158228,0.237342,4.193038,0.000000,0.000000,0.158228,1.661392,0.949367,0.000000,5.300633,0.000000
Norm,0.000000,41.701964,0.231036,25.067385,0.000000,0.077012,0.000000,0.616095,3.388525,1.925298,0.115518,17.096650,0.00000,0.231036,0.000000,0.269542,0.885637,0.000000,0.038506,0.154024,0.924143,0.770119,0.077012,6.430497,0.000000
not_measured,0.022647,62.633049,0.868121,13.612786,0.005392,0.441070,0.051764,2.114765,2.071628,1.175468,0.108920,12.956033,0.06794,0.002157,0.147742,0.001078,1.640264,0.008627,0.001078,0.419502,0.334307,0.627635,0.001078,0.683713,0.003235


admmission_source,Clinic Referral,Court/Law Enforcement,Emergency Room,Extramural Birth,HMO Referral,Normal Delivery,Not Available,Not Mapped,Physician Referral,Sick Baby,Transfer from Ambulatory Surgery Center,Transfer from a Skilled Nursing Facility (SNF),Transfer from a hospital,Transfer from another health care facility,Transfer from critial access hospital,Transfer from hospital inpt/same fac reslt in a sep claim
max_glu_serum,,,,,,,,,,,,,,,,
>200,2.011494,0.000000,83.908046,0.000000,0.000000,0.000000,0.000000,0.000000,14.080460,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
>300,0.378072,0.000000,91.493384,0.000000,0.000000,0.000000,0.000000,0.000000,7.939509,0.000000,0.000000,0.000000,0.189036,0.000000,0.000000,0.000000
Norm,2.061856,0.000000,75.773196,0.000000,0.000000,0.000000,0.000000,0.000000,22.164948,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
not_measured,1.157967,0.017108,60.172572,0.002138,0.199944,0.002138,0.133653,0.172145,31.376302,0.001069,0.002138,0.914184,3.406539,2.420717,0.008554,0.012831


Posmatranjem promenljive max_glu_serum, ukljucujuci i kategoriju not measured, mogu se uociti odredjeni obrasci u odnosu na druge atribute. Ova promenljiva pruza informaciju o tome da li su kod pacijenta utvrdjene promene nivoa glukoze u serumu, kao i o kategoriji izmerene vrednosti. Uocava se da pacijenti sa normalnim nivoom max_glu_serum najcesce pripadaju kategoriji pacijenata koji nisu ponovo hospitalizovani.

In [44]:
table_with_missing = pd.crosstab(data["max_glu_serum"],data["readmitted"])
chi2_with, p_value_with, dof, expected = chi2_contingency(table_with_missing)
print(f"chi2={chi2_with}, p_value={p_value_with}, dof= {dof},\n expected= {expected}")

chi2=51.92209834623324, p_value=1.9333043719858424e-09, dof= 6,
 expected= [[  165.7247509    518.68330287   800.59194623]
 [  141.06133679   441.49205039   681.44661282]
 [  289.82301555   907.08453708  1400.09244738]
 [10760.39089676 33677.74010966 51981.86899357]]


Kako bi se proverilo da li su uoceni obrasci izmedju promenljivih max_glu_serum i readmitted statisticki znacajni, primenjen je hi-kvadrat test nezavisnosti. U test je ukljucena i kategorija not measured, buduci da ona predstavlja informaciju o tome da vrednost max_glu_serum kod pacijenta nije bila izmerena. Dobijena je vrednost hi-kvadrat statistike od 51,92 uz 6 stepeni slobode i p-vrednost 1,93 x 10^-9. Kako je p-vrednost manja od nivoa znacajnosti od 0,05, odbacuje se nulta hipoteza o nezavisnosti promenljivih. Na osnovu toga moze se zakljuciti da postoji statisticki znacajna povezanost izmedju promenljivih max_glu_serum i readmitted.

In [45]:
crosstab_without_missing = pd.crosstab(data[data["max_glu_serum"]!="not_measured"]["max_glu_serum"],
                                 data[data["max_glu_serum"]!="not_measured"]["readmitted"], normalize="index"  )*100
print(crosstab_without_missing)
table_without_missing = pd.crosstab(data[data["max_glu_serum"]!="not_measured"]["max_glu_serum"],
                                 data[data["max_glu_serum"]!="not_measured"]["readmitted"])

readmitted           <30        >30         NO
max_glu_serum                                 
>200           12.457912  35.690236  51.851852
>300           14.319620  41.218354  44.462025
Norm           11.359261  34.077782  54.562957


In [46]:
chi2_without, p_value_without, dof, expected = chi2_contingency(table_without_missing)
print(f"chi2={chi2_without}, p_value={p_value_without}, dof= {dof},\n expected= {expected}")

chi2=35.11008068370349, p_value=4.40965642294135e-07, dof= 4,
 expected= [[ 183.61111111  537.77777778  763.61111111]
 [ 156.28582117  457.74485597  649.96932286]
 [ 321.10306771  940.47736626 1335.41956603]]


Posmatranjem raspodele pacijenata prema kategorijama max_glu_serum i readmitted uocava se da pacijenti sa normalnim nivoom glukoze imaju najveci udeo pacijenata koji nisu ponovo hospitalizovani, sa 54,56%. Kod pacijenata sa vrednostima vecim od 200 ovaj udeo iznosi 51,85%, dok kod pacijenata sa vrednostima vecim od 300 iznosi 44,46%. Takodje, u kategoriji >300 uocava se veci udeo ponovne hospitalizacije u periodu duzem od 30 dana, koji iznosi 41,22%.

In [47]:
n_with = table_with_missing.sum().sum()
r_with, k_with = table_with_missing.shape

cramers_v_with = (chi2_with / (n_with * min(r_with - 1, k_with - 1))) ** 0.5
n_without = table_without_missing.sum().sum()
r_without, k_without = table_without_missing.shape

cramers_v_without = (chi2_without / (n_without * min(r_without - 1, k_without - 1))) ** 0.5
print(f"With not_measured: Cramer's V = {cramers_v_with}")
print(f"Without not_measured: Cramer's V = {cramers_v_without}")

With not_measured: Cramer's V = 0.015972017141724526
Without not_measured: Cramer's V = 0.05730419799383266


Kako bi se dodatno ispitala jacina povezanosti izmedju promenljivih max_glu_serum i readmitted, izracunata je Cramer's V statistika. Kada je u analizu ukljucena kategorija not measured, dobijena vrednost Cramer's V iznosi 0,016. Kada se kategorija not measured izostavi, vrednost Cramer's V iznosi 0,057.

Dobijene vrednosti ukazuju da je povezanost izmedju promenljivih max_glu_serum i readmitted veoma slaba, iako je hi-kvadrat test pokazao da je ona statisticki znacajna. To pokazuje da statisticka znacajnost povezanosti ne podrazumeva i njenu veliku prakticnu jacinu. Uklanjanjem kategorije not measured vrednost Cramer's V raste, ali povezanost i dalje ostaje veoma slaba.

Na osnovu sprovedene analize, promenljiva max_glu_serum se za sada zadrzava u skupu podataka. Iako je jacina povezanosti sa promenljivom readmitted veoma slaba, hi-kvadrat test je pokazao statisticki znacajnu povezanost, a deskriptivnom analizom su uoceni odredjeni obrasci izmedju kategorija max_glu_serum i ponovne hospitalizacije. Zbog toga se u ovoj fazi promenljiva ne uklanja, vec se ostavlja za dalju analizu i procenu njenog znacaja u procesu izgradnje modela.

In [48]:
data_overview(data)

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,...,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type,discharge_disposition,admmission_source
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,not_measured,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,NaN,Not Mapped,Physician Referral
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,not_measured,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,Emergency,Discharged to home,Emergency Room
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,not_measured,NaN,No,No,...,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,not_measured,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,not_measured,NaN,No,No,...,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
5,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,NaN,NaN,31,6,16,0,0,0,414,411,250,9,not_measured,NaN,No,No,...,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30,Urgent,Discharged to home,Clinic Referral
6,55842,84259809,Caucasian,Male,[60-70),3,1,2,4,NaN,NaN,70,1,21,0,0,0,414,411,V45,7,not_measured,NaN,Steady,No,...,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Elective,Discharged to home,Clinic Referral
7,63768,114882984,Caucasian,Male,[70-80),1,1,7,5,NaN,NaN,73,0,12,0,0,0,428,492,250,8,not_measured,NaN,No,No,...,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30,Emergency,Discharged to home,Emergency Room
8,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,NaN,NaN,68,2,28,0,0,0,398,427,38,8,not_measured,NaN,No,No,...,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Urgent,Discharged to home,Transfer from a hospital
9,15738,63555939,Caucasian,Female,[90-100),3,3,4,12,NaN,InternalMedicine,33,3,18,0,0,0,434,198,486,8,not_measured,NaN,No,No,...,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Elective,Discharged/transferred to SNF,Transfer from a hospital


Number of rows:101766
Number of columns:52
                         Data type:  Missing values:  Percentage of na:  \
encounter_id                  int64                0           0.000000   
patient_nbr                   int64                0           0.000000   
race                         object             2273           2.233555   
gender                       object                0           0.000000   
age                          object                0           0.000000   
admission_type_id             int64                0           0.000000   
discharge_disposition_id      int64                0           0.000000   
admission_source_id           int64                0           0.000000   
time_in_hospital              int64                0           0.000000   
payer_code                   object            40256          39.557416   
medical_specialty            object            49949          49.082208   
num_lab_procedures            int64                0     

In [49]:
for col in data[["admission_type","discharge_disposition","admmission_source"]].columns:
     data.loc[data[col].isna(), col] = "Missing"
            

In [50]:
data["admission_type_id"] = data["admission_type"]
data["discharge_disposition_id"] = data["discharge_disposition"]
data["admission_source_id"] = data["admmission_source"]

In [51]:
data.drop(data[["admission_type","discharge_disposition","admmission_source"]].columns, axis= 1, inplace = True)

Vrednosti NULL u atributima admission_type, discharge_disposition i admmission_source pretvorene su u nedostajuce vrednosti, posto za njih nije bilo dostupno konkretno znacenje tipa prijema. Pri tome su kategorije Unknown, Not Mapped i Not Available zadrzane odvojeno, jer predstavljaju zasebne vrednosti u mapping tabeli. Nedostajuce vrednosti su zatim oznacene kao Missing, cime su zadrzani svi redovi u skupu podataka.

In [52]:
data["payer_code"].describe()

count     61510
unique       17
top          MC
freq      32439
Name: payer_code, dtype: object

In [53]:
data.drop(columns="payer_code", inplace=True)

Atribut payer_code je uklonjen iz skupa podataka zbog visokog udela nedostajucih vrednosti, koji iznosi oko 40%. Imputacija nedostajucih vrednosti najcescom kategorijom ne bi bila opravdana, jer bi se veliki broj nepoznatih vrednosti svrstao u kategoriju MC, koja je najzastupljenija u atributu, bez osnova da se pretpostavi da su te vrednosti zaista pripadale toj kategoriji. Dodatno, atribut opisuje nacin finansiranja zdravstvene usluge, a ne direktnu karakteristiku zdravstvenog stanja ili toka hospitalizacije, zbog cega je procenjeno da njegova potencijalna korist za analizu readmisije nije dovoljna da opravda zadrzavanje atributa sa tolikim udelom nedostajucih podataka.

In [54]:
data["A1Cresult"] = data["A1Cresult"].fillna("Not measured")

Atribut A1Cresult je zadrzan jer predstavlja rezultat jednog od dva laboratorijska merenja dostupna u skupu podataka, pored atributa max_glu_serum. Iako su tokom hospitalizacije pacijentima obavljane brojne laboratorijske procedure, skup podataka ne sadrzi rezultate svih izvedenih analiza, vec su od pojedinacnih laboratorijskih rezultata dostupni upravo HbA1c i maksimalna vrednost glukoze u serumu. S obzirom na to da je analiza usmerena na pacijente sa dijabetesom i njihovu readmisiju, rezultat HbA1c predstavlja relevantan atribut za dalju analizu. Nedostajuce vrednosti su zato izdvojene u posebnu kategoriju Not measured, cime je sacuvana informacija da HbA1c merenje nije evidentirano.

In [55]:
data["medical_specialty"] = data["medical_specialty"].fillna("Missing")

Atribut medical_specialty je zadrzan jer predstavlja specijalnost lekara ukljucenog u hospitalizaciju i moze pruziti informaciju o vrsti i slozenosti zdravstvenog stanja pacijenta, sto ga cini potencijalno relevantnim za analizu readmisije. Zbog visokog udela nedostajucih vrednosti, one su izdvojene u posebnu kategoriju Missing, cime je izbegnuto proizvoljno pripisivanje konkretne medicinske specijalnosti.

In [56]:
data["discharge_disposition_id"].value_counts()

discharge_disposition_id
Discharged to home                                                                                           60234
Discharged/transferred to SNF                                                                                13954
Discharged/transferred to home with home health service                                                      12902
Missing                                                                                                       3691
Discharged/transferred to another short term hospital                                                         2128
Discharged/transferred to another rehab fac including rehab units of a hospital .                             1993
Expired                                                                                                       1642
Discharged/transferred to another type of inpatient care institution                                          1184
Not Mapped                                             

In [57]:
remove_disposition = [
    "Expired",
    "Hospice / home",
    "Hospice / medical facility",
    "Expired at home. Medicaid only, hospice.",
    "Expired in a medical facility. Medicaid only, hospice."
]

data["discharge_disposition_id"].isin(remove_disposition).sum()

np.int64(2423)

In [58]:
data = data[
    ~data["discharge_disposition_id"].isin(remove_disposition)
]

In [59]:
data_overview(data)

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),Missing,Not Mapped,Physician Referral,1,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,not_measured,Not measured,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),Emergency,Discharged to home,Emergency Room,3,Missing,59,0,18,0,0,0,276,250.01,255,9,not_measured,Not measured,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),Emergency,Discharged to home,Emergency Room,2,Missing,11,5,13,2,0,1,648,250,V27,6,not_measured,Not measured,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),Emergency,Discharged to home,Emergency Room,2,Missing,44,1,16,0,0,0,8,250.43,403,7,not_measured,Not measured,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),Emergency,Discharged to home,Emergency Room,1,Missing,51,0,8,0,0,0,197,157,250,5,not_measured,Not measured,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
5,35754,82637451,Caucasian,Male,[50-60),Urgent,Discharged to home,Clinic Referral,3,Missing,31,6,16,0,0,0,414,411,250,9,not_measured,Not measured,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
6,55842,84259809,Caucasian,Male,[60-70),Elective,Discharged to home,Clinic Referral,4,Missing,70,1,21,0,0,0,414,411,V45,7,not_measured,Not measured,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
7,63768,114882984,Caucasian,Male,[70-80),Emergency,Discharged to home,Emergency Room,5,Missing,73,0,12,0,0,0,428,492,250,8,not_measured,Not measured,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
8,12522,48330783,Caucasian,Female,[80-90),Urgent,Discharged to home,Transfer from a hospital,13,Missing,68,2,28,0,0,0,398,427,38,8,not_measured,Not measured,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,15738,63555939,Caucasian,Female,[90-100),Elective,Discharged/transferred to SNF,Transfer from a hospital,12,InternalMedicine,33,3,18,0,0,0,434,198,486,8,not_measured,Not measured,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Number of rows:99343
Number of columns:48
                         Data type:  Missing values:  Percentage of na:  \
encounter_id                  int64                0           0.000000   
patient_nbr                   int64                0           0.000000   
race                         object             2234           2.248774   
gender                       object                0           0.000000   
age                          object                0           0.000000   
admission_type_id            object                0           0.000000   
discharge_disposition_id     object                0           0.000000   
admission_source_id          object                0           0.000000   
time_in_hospital              int64                0           0.000000   
medical_specialty            object                0           0.000000   
num_lab_procedures            int64                0           0.000000   
num_procedures                int64                0      

Uklonjeni su redovi kod kojih je ishod hospitalizacije bio Expired ili Hospice, jer kod ovih pacijenata readmisija nije moguca ili nije uporediva sa ostalim slucajevima. Ukupno je uklonjeno 2.423 redova.

### Selekcija atrbitua

In [60]:
data.drop("encounter_id", axis= 1, inplace=True)

Atribut "encounter_id" predstavlja jedinstveni identifikator, pa je zbog toga uklonjen iz dalje analize. 

In [61]:
data["patient_nbr"].nunique()

69990

Atribut patient_nbr se za sada zadrzava jer se koristi za grupisanje hospitalizacija istog pacijenta prilikom podele podataka na skupove za treniranje i testiranje. Nece biti koriscen kao ulazni atribut modela.

In [62]:
data.nunique().sort_values()

citoglipton                     1
examide                         1
diabetesMed                     2
change                          2
acetohexamide                   2
metformin-pioglitazone          2
metformin-rosiglitazone         2
glimepiride-pioglitazone        2
glipizide-metformin             2
tolbutamide                     2
troglitazone                    2
readmitted                      3
gender                          3
tolazamide                      3
insulin                         4
miglitol                        4
acarbose                        4
rosiglitazone                   4
pioglitazone                    4
glyburide-metformin             4
glyburide                       4
glipizide                       4
glimepiride                     4
chlorpropamide                  4
nateglinide                     4
metformin                       4
A1Cresult                       4
max_glu_serum                   4
repaglinide                     4
race          

In [63]:
data = data.drop(["citoglipton", "examide"], axis=1)

Atributi citoglipton i examide su uklonjeni jer imaju samo jednu jedinstvenu vrednost u celom skupu podataka i samim tim ne sadrze korisnu informaciju za model.

In [64]:
data[["acetohexamide","metformin-pioglitazone","metformin-rosiglitazone","glimepiride-pioglitazone","glipizide-metformin","tolbutamide","troglitazone"]].apply(pd.Series.value_counts)

,acetohexamide,metformin-pioglitazone,metformin-rosiglitazone,glimepiride-pioglitazone,glipizide-metformin,tolbutamide,troglitazone
No,99342,99342,99341,99342,99330,99322,99340
Steady,1,1,2,1,13,21,3


Kod navedenih atributa uocena je izrazito mala varijabilnost, pri cemu kategorija „Steady“ cini svega 0,001–0,02% vrednosti, dok gotovo sve preostale vrednosti pripadaju kategoriji „No“. Zadrzavanje ovakvih atributa moglo bi dovesti do nestabilnog ucenja i overfittinga, jer bi model mogao da pridaje preveliki znacaj veoma malom broju primera iz retke kategorije, cime bi se smanjila sposobnost generalizacije na nove podatke.

In [65]:
data = data.drop([
    "acetohexamide",
    "metformin-pioglitazone",
    "metformin-rosiglitazone",
    "glimepiride-pioglitazone",
    "glipizide-metformin",
    "tolbutamide",
    "troglitazone"
], axis=1)

In [66]:
data[["tolazamide","repaglinide","metformin","glimepiride","glipizide","glyburide","pioglitazone","acarbose","nateglinide","miglitol","insulin","glyburide-metformin","rosiglitazone","chlorpropamide"]].apply(pd.Series.value_counts)

,tolazamide,repaglinide,metformin,glimepiride,glipizide,glyburide,pioglitazone,acarbose,nateglinide,miglitol,insulin,glyburide-metformin,rosiglitazone,chlorpropamide
Down,NaN,43,574,191,548,560,117,3,11,5,11908,6,85,1
No,99304.0,97825,79499,94221,86812,88820,92088,99038,98654,99305,46379,98645,93039,99258
Steady,38.0,1368,18207,4609,11219,9162,6908,292,654,31,30069,684,6048,78
Up,1.0,107,1063,322,764,801,230,10,24,2,10987,8,171,6


In [67]:
data = data.drop(["tolazamide","miglitol","chlorpropamide"], axis= 1)

Atributi tolazamide, miglitol i chlorpropamide uklonjeni su zbog izrazito male varijabilnosti, odnosno veoma malog broja vrednosti koje se razlikuju od kategorije No, sto ogranicava njihov potencijalni doprinos modelu.

In [68]:
drug_columns = [
    "repaglinide",
    "acarbose",
    "nateglinide",
    "glyburide-metformin"
]

for column in drug_columns:
    display(
        pd.crosstab(
            data[column],
            data["readmitted"],
            normalize="index"
        ) * 100
    )

readmitted,<30,>30,NO
repaglinide,,,
Down,6.976744,44.186047,48.837209
No,11.355993,35.637107,53.006900
Steady,13.304094,42.982456,43.713450
Up,18.691589,30.841121,50.467290


readmitted,<30,>30,NO
acarbose,,,
Down,33.333333,33.333333,33.333333
No,11.395626,35.690341,52.914033
Steady,8.561644,51.369863,40.068493
Up,20.000000,40.000000,40.000000


readmitted,<30,>30,NO
nateglinide,,,
Down,9.090909,36.363636,54.545455
No,11.387273,35.720802,52.891925
Steady,11.926606,37.920489,50.152905
Up,4.166667,41.666667,54.166667


readmitted,<30,>30,NO
glyburide-metformin,,,
Down,16.666667,0.000000,83.333333
No,11.390339,35.728116,52.881545
Steady,11.257310,37.573099,51.169591
Up,0.000000,12.500000,87.500000


In [69]:
for column in drug_columns:
    table = pd.crosstab(data[column], data["readmitted"])
    chi2, p_value, dof, expected = chi2_contingency(table)

    print(f"{column}:")
    print(f"Chi-square = {chi2:.2f}")
    print(f"p-value = {p_value:.4f}")
    n = table.to_numpy().sum()
    r, k = table.shape

    cramer_v = np.sqrt(
        chi2 / (n * min(r - 1, k - 1))
    )

    print(f"{column}: Cramer's V = {cramer_v:.4f}")

repaglinide:
Chi-square = 54.48
p-value = 0.0000
repaglinide: Cramer's V = 0.0166
acarbose:
Chi-square = 33.67
p-value = 0.0000
acarbose: Cramer's V = 0.0130
nateglinide:
Chi-square = 3.37
p-value = 0.7613
nateglinide: Cramer's V = 0.0041
glyburide-metformin:
Chi-square = 8.31
p-value = 0.2161
glyburide-metformin: Cramer's V = 0.0065


Za preostale atribute terapije koji pokazuju relativno slabu varijabilnost ispitana je povezanost sa readmitted pomocu Chi-square testa i Cramerovog testa. Iako su repaglinide i acarbose pokazali statisticki znacajnu povezanost, vrednosti Cramerovog testa bile su veoma male, sto ukazuje na veoma slabu prakticnu povezanost. Kod nateglinide i glyburide-metformin nije utvrdjena statisticki znacajna povezanost, uz takodje veoma male vrednosti Cramerovog testa. Na osnovu ovih rezultata, sva cetiri atributa su uklonjena iz dalje analize.

In [70]:
data = data.drop(["repaglinide","acarbose","nateglinide","glyburide-metformin"], axis = 1)

In [71]:
data_overview(data)

,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,insulin,change,diabetesMed,readmitted
0,8222157,Caucasian,Female,[0-10),Missing,Not Mapped,Physician Referral,1,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,not_measured,Not measured,No,No,No,No,No,No,No,No,No,NO
1,55629189,Caucasian,Female,[10-20),Emergency,Discharged to home,Emergency Room,3,Missing,59,0,18,0,0,0,276,250.01,255,9,not_measured,Not measured,No,No,No,No,No,No,Up,Ch,Yes,>30
2,86047875,AfricanAmerican,Female,[20-30),Emergency,Discharged to home,Emergency Room,2,Missing,11,5,13,2,0,1,648,250,V27,6,not_measured,Not measured,No,No,Steady,No,No,No,No,No,Yes,NO
3,82442376,Caucasian,Male,[30-40),Emergency,Discharged to home,Emergency Room,2,Missing,44,1,16,0,0,0,8,250.43,403,7,not_measured,Not measured,No,No,No,No,No,No,Up,Ch,Yes,NO
4,42519267,Caucasian,Male,[40-50),Emergency,Discharged to home,Emergency Room,1,Missing,51,0,8,0,0,0,197,157,250,5,not_measured,Not measured,No,No,Steady,No,No,No,Steady,Ch,Yes,NO
5,82637451,Caucasian,Male,[50-60),Urgent,Discharged to home,Clinic Referral,3,Missing,31,6,16,0,0,0,414,411,250,9,not_measured,Not measured,No,No,No,No,No,No,Steady,No,Yes,>30
6,84259809,Caucasian,Male,[60-70),Elective,Discharged to home,Clinic Referral,4,Missing,70,1,21,0,0,0,414,411,V45,7,not_measured,Not measured,Steady,Steady,No,No,No,No,Steady,Ch,Yes,NO
7,114882984,Caucasian,Male,[70-80),Emergency,Discharged to home,Emergency Room,5,Missing,73,0,12,0,0,0,428,492,250,8,not_measured,Not measured,No,No,No,Steady,No,No,No,No,Yes,>30
8,48330783,Caucasian,Female,[80-90),Urgent,Discharged to home,Transfer from a hospital,13,Missing,68,2,28,0,0,0,398,427,38,8,not_measured,Not measured,No,No,Steady,No,No,No,Steady,Ch,Yes,NO
9,63555939,Caucasian,Female,[90-100),Elective,Discharged/transferred to SNF,Transfer from a hospital,12,InternalMedicine,33,3,18,0,0,0,434,198,486,8,not_measured,Not measured,No,No,No,No,No,Steady,Steady,Ch,Yes,NO


Number of rows:99343
Number of columns:31
                         Data type:  Missing values:  Percentage of na:  \
patient_nbr                   int64                0           0.000000   
race                         object             2234           2.248774   
gender                       object                0           0.000000   
age                          object                0           0.000000   
admission_type_id            object                0           0.000000   
discharge_disposition_id     object                0           0.000000   
admission_source_id          object                0           0.000000   
time_in_hospital              int64                0           0.000000   
medical_specialty            object                0           0.000000   
num_lab_procedures            int64                0           0.000000   
num_procedures                int64                0           0.000000   
num_medications               int64                0      

In [72]:
medical_data = data[
    (data['medical_specialty'] != 'Missing') &
    (data['admission_type_id'] != 'Missing')
].copy()

In [73]:
table = pd.crosstab(
    medical_data['medical_specialty'],
    medical_data['admission_type_id']
)
print(table)

admission_type_id                     Elective  Emergency  Newborn  \
medical_specialty                                                    
AllergyandImmunology                         0          0        0   
Anesthesiology                               6          3        0   
Anesthesiology-Pediatric                     2         12        0   
Cardiology                                1711       1604        0   
Cardiology-Pediatric                         0          0        0   
DCPTEAM                                      3          2        0   
Dentistry                                    0          1        0   
Dermatology                                  0          0        0   
Emergency/Trauma                           877       2389        0   
Endocrinology                                4         71        0   
Endocrinology-Metabolism                     1          7        0   
Family/GeneralPractice                    1050       3384        1   
Gastroenterology    

In [74]:
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape
    
    return np.sqrt((chi2 / n) / min(r - 1, k - 1))

In [75]:
cramers_v(
    medical_data['medical_specialty'],
    medical_data['admission_type_id']
)

np.float64(0.3055287396546277)

In [76]:
medical_data = data[
    (data['medical_specialty'] != 'Missing') &
    (data['discharge_disposition_id'] != 'Missing')
].copy()

In [77]:
cramers_v(
    medical_data['medical_specialty'],
    medical_data['discharge_disposition_id']
)

np.float64(0.09039242358157046)

In [78]:
medical_data = data[
    (data['medical_specialty'] != 'Missing') &
    (data['admission_source_id'] != 'Missing')
].copy()

In [79]:
cramers_v(
    medical_data['medical_specialty'],
    medical_data['admission_source_id']
)

np.float64(0.20980333671833734)

In [80]:
medical_data = data[
    (data['medical_specialty'] != 'Missing') &
    (data['diag_1'].notna())
].copy()

In [81]:
cramers_v(
    medical_data['medical_specialty'],
    medical_data['diag_1']
)

np.float64(0.3115135803891224)

In [82]:
medical_data = data[
    (data['medical_specialty'] != 'Missing') &
    (data['diag_2'].notna())
].copy()

In [83]:
cramers_v(
    medical_data['medical_specialty'],
    medical_data['diag_2']
)

np.float64(0.22736980919190505)

In [84]:
medical_data = data[
    (data['medical_specialty'] != 'Missing') &
    (data['diag_3'].notna())
].copy()

In [85]:
cramers_v(
    medical_data['medical_specialty'],
    medical_data['diag_3']
)

np.float64(0.20704981486034818)

Proverena je povezanost promenljive medical_specialty sa ostalim kategorijskim atributima koji bi potencijalno mogli sadržati sličnu informaciju. Za procenu stepena povezanosti korišćen je Cramer's V. Dobijene vrednosti nisu ukazale na jaku povezanost između medical_specialty i analiziranih atributa, zbog čega nije uočena značajna redundantnost informacija. Na osnovu ovog rezultata, medical_specialty nije eliminisana zbog potencijalnog dupliranja informacija sa drugim atributima.

In [86]:
table = pd.crosstab(
    data['medical_specialty'],
    data['readmitted']
)

table

readmitted,<30,>30,NO
medical_specialty,,,
AllergyandImmunology,3,3,1
Anesthesiology,1,3,8
Anesthesiology-Pediatric,1,5,13
Cardiology,424,1853,3002
Cardiology-Pediatric,1,4,2
DCPTEAM,0,1,4
Dentistry,0,3,1
Dermatology,0,1,0
Emergency/Trauma,845,3004,3570


In [87]:
chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p_value)

Chi-square: 1471.5740613189244
p-value: 1.2749422741591935e-218


In [88]:
n = table.sum().sum()
r, k = table.shape

cramers_v_value = np.sqrt(
    (chi2 / n) / min(r - 1, k - 1)
)

print("Cramer's V:", cramers_v_value)

Cramer's V: 0.08606120622375836


In [89]:
pd.crosstab(
    data['medical_specialty'],
    data['readmitted'],
    normalize='index'
).round(3)

readmitted,<30,>30,NO
medical_specialty,,,
AllergyandImmunology,0.429,0.429,0.143
Anesthesiology,0.083,0.250,0.667
Anesthesiology-Pediatric,0.053,0.263,0.684
Cardiology,0.080,0.351,0.569
Cardiology-Pediatric,0.143,0.571,0.286
DCPTEAM,0.000,0.200,0.800
Dentistry,0.000,0.750,0.250
Dermatology,0.000,1.000,0.000
Emergency/Trauma,0.114,0.405,0.481


Analizom povezanosti sa ciljnom promenljivom readmitted utvrđena je statistički značajna povezanost (χ² test, p < 0.001), ali je intenzitet povezanosti veoma slab (Cramer's V = 0.086). Iako je uticaj promenljive na ciljnu promenljivu slab, zbog postojanja statistički značajne povezanosti odlučeno je da se medical_specialty zadrži u daljoj analizi i modeliranju.

In [90]:
compare_column_with_all_columns("readmitted")


--- patient_nbr ---


,count,mean,std,min,25%,50%,75%,max
readmitted,,,,,,,,
<30,11314.0,5.520815e+07,3.780236e+07,135.0,23583822.75,48139200.0,88227540.0,188970179.0
>30,35502.0,5.814380e+07,3.661933e+07,135.0,24800184.00,57902751.0,89308872.0,189481478.0
NO,52527.0,5.143300e+07,4.006361e+07,378.0,20185015.50,42050817.0,85699066.5,189502619.0


race,AfricanAmerican,Asian,Caucasian,Hispanic,Other
readmitted,,,,,
<30,19.315118,0.584217,76.900953,1.905447,1.294266
>30,18.928040,0.460474,77.502574,1.833314,1.275598
NO,19.610341,0.787942,75.591446,2.281503,1.728768


gender,Female,Male,Unknown/Invalid
readmitted,,,
<30,54.162984,45.837016,0.000000
>30,54.906766,45.093234,0.000000
NO,52.987987,47.006302,0.005711


age,[0-10),[10-20),[20-30),[30-40),[40-50),[50-60),[60-70),[70-80),[80-90),[90-100)
readmitted,,,,,,,,,,
<30,0.026516,0.353544,2.085911,3.747569,9.050734,14.733958,22.034647,26.993106,18.251724,2.722291
>30,0.073235,0.630950,1.436539,3.343474,9.224832,16.649766,22.218467,26.654836,17.494789,2.273111
NO,0.249396,0.811011,1.719116,4.098844,10.105279,18.051669,22.232376,24.395073,15.531060,2.806176


...

pioglitazone,Down,No,Steady,Up
readmitted,,,,
<30,0.159095,93.176595,6.407990,0.256320
>30,0.126753,92.237057,7.377049,0.259140
NO,0.102804,92.904601,6.785082,0.207512


rosiglitazone,Down,No,Steady,Up
readmitted,,,,
<30,0.044193,94.131165,5.656709,0.167934
>30,0.061968,93.217284,6.574278,0.146471
NO,0.110419,93.846974,5.852228,0.190378


insulin,Down,No,Steady,Up
readmitted,,,,
<30,14.963762,41.859643,30.228036,12.948559
>30,13.362627,44.887612,29.485663,12.264098
NO,10.415596,48.940545,30.805110,9.838750


change,Ch,No
readmitted,,
<30,49.010076,50.989924
>30,48.591629,51.408371
NO,44.407638,55.592362


diabetesMed,No,Yes
readmitted,,
<30,19.736610,80.263390
>30,20.334066,79.665934
NO,25.076627,74.923373


### Analiza kategorijskih atributa

Kategorijski atributi analizirani su kroz raspodelu njihovih vrednosti u odnosu na ciljnu promenljivu `readmitted`. Kod pojedinih atributa uoceni su izrazeniji obrasci izmedju grupa, dok su kod drugih raspodele bile veoma slicne.

Za detaljniju procenu povezanosti odabranih kategorijskih atributa sa `readmitted` primenjuju se hi-kvadrat test i Cramerov V koeficijent.

### Zakljucak analize kategorijskih atributa

Kod `medical_specialty` utvrdjena je statisticki znacajna, ali slaba povezanost sa `readmitted` (Cramerov V = 0.086), dok je kod `max_glu_serum` utvrdjena statisticki znacajna, ali veoma slaba povezanost (Cramerov V = 0.016). Na osnovu rezultata ove analize atributi nisu automatski eliminisani, vec su zadrzani za naredne faze obrade i modelovanja.

### Analiza numerickih atributa

Na osnovu deskriptivne analize izdvojeni su numericki atributi kod kojih su uocene izrazenije razlike izmedju grupa `readmitted`: `time_in_hospital`, `num_medications`, `number_outpatient`, `number_emergency` i `number_inpatient`.

Za ispitivanje statisticke znacajnosti razlika izmedju tri nezavisne grupe primenjuje se Kruskal-Wallis test.

In [94]:
numerical_columns = [
    'time_in_hospital',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

for col in numerical_columns:
    groups = [
        data.loc[data['readmitted'] == category, col].dropna()
        for category in ['<30', '>30', 'NO']
    ]
    
    stat, p = kruskal(*groups)
    
    print(f"{col}:")
    print(f"Kruskal-Wallis statistic = {stat:.4f}")
    print(f"p-value = {p:.4e}")
    print()

time_in_hospital:
Kruskal-Wallis statistic = 522.6134
p-value = 3.2804e-114

num_medications:
Kruskal-Wallis statistic = 627.8269
p-value = 4.6679e-137

number_outpatient:
Kruskal-Wallis statistic = 1276.2914
p-value = 7.1917e-278

number_emergency:
Kruskal-Wallis statistic = 1692.8219
p-value = 0.0000e+00

number_inpatient:
Kruskal-Wallis statistic = 5972.5782
p-value = 0.0000e+00



Kruskal-Wallis testom utvrdjeno je da za svih pet analiziranih numerickih atributa postoje statisticki znacajne razlike izmedju grupa ciljne promenljive readmitted (p < 0.001). Na osnovu toga, ovi atributi nisu eliminisani u ovoj fazi analize, vec su zadrzani za dalju obradu i modelovanje.

In [96]:
data[
    ['num_lab_procedures',
     'num_procedures',
     'num_medications',
     'time_in_hospital',
     'number_outpatient',
     'number_emergency',
     'number_inpatient',
    'number_diagnoses']
].corr(method='spearman')

,num_lab_procedures,num_procedures,num_medications,time_in_hospital,number_outpatient,number_emergency,number_inpatient,number_diagnoses
num_lab_procedures,1.000000,0.016508,0.248416,0.338932,-0.023744,0.008058,0.041246,0.165544
num_procedures,0.016508,1.000000,0.348619,0.186796,-0.023753,-0.046510,-0.064419,0.063339
num_medications,0.248416,0.348619,1.000000,0.463446,0.075430,0.045545,0.100765,0.293927
time_in_hospital,0.338932,0.186796,0.463446,1.000000,-0.013686,-0.000782,0.093031,0.237700
number_outpatient,-0.023744,-0.023753,0.075430,-0.013686,1.000000,0.179225,0.156665,0.113742
number_emergency,0.008058,-0.046510,0.045545,-0.000782,0.179225,1.000000,0.222692,0.093217
number_inpatient,0.041246,-0.064419,0.100765,0.093031,0.156665,0.222692,1.000000,0.136186
number_diagnoses,0.165544,0.063339,0.293927,0.237700,0.113742,0.093217,0.136186,1.000000


Analizom korelacija izmedju numerickih atributa nije utvrdjena jaka povezanost izmedju posmatranih promenljivih. Na osnovu toga nije uocena izrazenija redundantnost medju numerickim atributima, odnosno nema osnova za eliminaciju atributa zbog dupliranja informacija. Numericki atributi su stoga zadrzani za dalju analizu i modelovanje.

In [97]:
data[['diag_1', 'diag_2', 'diag_3']].nunique()

diag_1    715
diag_2    747
diag_3    786
dtype: int64